In [1]:
import pandas as pd
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses
from head_tail import extract_head_tail

c:\projects\ml_learn\HackRush\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def train_bi_encoder(train_csv_path, output_model_path):
    print("Loading training data...")
    df = pd.read_csv(train_csv_path)
    texts = df['text'].tolist()
    
    # 1. Extract Heads and Tails
    heads = []
    tails = []
    for text in texts:
        h, t = extract_head_tail(text)
        heads.append(h)
        tails.append(t)
        
    # 2. Create Positive Pairs (Tail of Page i matches Head of Page i+1)
    train_examples = []
    for i in range(len(texts) - 1):
        # InputExample takes the two texts that SHOULD be close in vector space
        example = InputExample(texts=[tails[i], heads[i+1]])
        train_examples.append(example)
        
    print(f"Generated {len(train_examples)} positive training pairs.")
    
    # 3. Load Base Model
    # We use all-MiniLM-L6-v2 as a fast starting point
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # 4. Setup DataLoader and Loss Function
    # Batch size is critical here. A larger batch size means more automatic negative examples per batch.
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
    train_loss = losses.MultipleNegativesRankingLoss(model=model)
    
    # 5. Fine-Tune the Model
    print("Starting Bi-Encoder training...")
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=3, # Keep epochs low to prevent overfitting on a single book
        warmup_steps=100,
        show_progress_bar=True
    )
    
    # 6. Save the newly fine-tuned model
    model.save(output_model_path)
    print(f"Fine-tuned Bi-Encoder saved to {output_model_path}")

In [3]:
train_bi_encoder('./data/Mysterious_Affair_at_Styles_Train_Data.csv', './models/fine_tuned_bi_encoder')

Loading training data...
Generated 206 positive training pairs.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9363.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting Bi-Encoder training...


c:\projects\ml_learn\HackRush\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.61it/s]


Fine-tuned Bi-Encoder saved to ./models/fine_tuned_bi_encoder
